In [1]:
import os
from collections import defaultdict

epa_dir   = "/mnt/2TB_WD/rishi/epa/epa_data_by_site_all_years"
cpcb_dir  = "/mnt/2TB_WD/rishi/cpcb/separated"
aurn_dir  = "/mnt/2TB_WD/rishi/aurn/aurn_processed"
cnemc_dir = "/mnt/2TB_WD/rishi/cnemc_data/by_site_pollutant"

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

def count_by_pollutant(folder):
    counts = defaultdict(int)
    for fname in os.listdir(folder):
        if not fname.endswith(".csv"):
            continue
        stem = fname[:-4]  # strip .csv
        for pol in POLLUTANTS:
            if stem.endswith(f"_{pol}"):
                counts[pol] += 1
                break
    return counts

epa_counts   = count_by_pollutant(epa_dir)
cpcb_counts  = count_by_pollutant(cpcb_dir)
aurn_counts  = count_by_pollutant(aurn_dir)
cnemc_counts = count_by_pollutant(cnemc_dir)

print(f"{'Pollutant':<12} {'EPA':>8} {'CPCB':>8} {'AURN':>8} {'CNEMC':>8}")
print("-" * 46)
for pol in POLLUTANTS:
    print(f"{pol:<12} {epa_counts[pol]:>8} {cpcb_counts[pol]:>8} {aurn_counts[pol]:>8} {cnemc_counts[pol]:>8}")


Pollutant         EPA     CPCB     AURN    CNEMC
----------------------------------------------
PM2.5             997      561      162     1738
PM10              522      556      150     1738
NO2               493      563      164     1738
SO2               443      541       29     1738
CO                263      558        7     1738
Ozone            1302      544      100     1738


In [3]:
import sys
import yaml
sys.path.insert(0, "/home/student/rishi")
from imputation import get_sites_per_pollutant

with open("/home/student/rishi/config.yaml") as f:
    config = yaml.safe_load(f)

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]
POL_KEYS = {
    "PM2.5": "PM2.5 (µg/m³)",
    "PM10":  "PM10 (µg/m³)",
    "NO2":   "NO2 (µg/m³)",
    "SO2":   "SO2 (µg/m³)",
    "CO":    "CO (mg/m³)",
    "Ozone": "Ozone (µg/m³)",
}

results = {}
for dataset in ["epa", "cpcb", "aurn", "cnemc"]:
    cfg = config[dataset]["imputation"]
    results[dataset] = get_sites_per_pollutant(
        dicts_dir=cfg["dicts_dir"],
        features=cfg["features"],
        max_gap_hours=cfg["max_gap_hours"],
        max_data_missing=cfg["max_data_missing"],
    )

print(f"{'Pollutant':<12} {'EPA':>8} {'CPCB':>8} {'AURN':>8} {'CNEMC':>8}")
print("-" * 46)
for pol in POLLUTANTS:
    key = POL_KEYS[pol]
    counts = [len(results[ds].get(key, [])) for ds in ["epa", "cpcb", "aurn", "cnemc"]]
    print(f"{pol:<12} {counts[0]:>8} {counts[1]:>8} {counts[2]:>8} {counts[3]:>8}")
print("-" * 46)
totals = [sum(len(v) for v in results[ds].values()) for ds in ["epa", "cpcb", "aurn", "cnemc"]]
print(f"{'Total':<12} {totals[0]:>8} {totals[1]:>8} {totals[2]:>8} {totals[3]:>8}")


PM2.5 (µg/m³): 445 valid sites
PM10 (µg/m³): 220 valid sites
NO2 (µg/m³): 212 valid sites
SO2 (µg/m³): 241 valid sites
CO (mg/m³): 91 valid sites
Ozone (µg/m³): 488 valid sites
PM2.5 (µg/m³): 190 valid sites
PM10 (µg/m³): 191 valid sites
NO2 (µg/m³): 179 valid sites
SO2 (µg/m³): 183 valid sites
CO (mg/m³): 179 valid sites
Ozone (µg/m³): 182 valid sites
PM2.5 (µg/m³): 61 valid sites
PM10 (µg/m³): 80 valid sites
NO2 (µg/m³): 70 valid sites
SO2 (µg/m³): 8 valid sites
CO (mg/m³): 1 valid sites
Ozone (µg/m³): 39 valid sites
PM2.5 (µg/m³): 1471 valid sites
PM10 (µg/m³): 1481 valid sites
NO2 (µg/m³): 1481 valid sites
SO2 (µg/m³): 1485 valid sites
CO (mg/m³): 1487 valid sites
Ozone (µg/m³): 1488 valid sites
Pollutant         EPA     CPCB     AURN    CNEMC
----------------------------------------------
PM2.5             445      190       61     1471
PM10              220      191       80     1481
NO2               212      179       70     1481
SO2               241      183        8     1485